Controlled transfer to analyze how much model is learning from each dataset

In [1]:
from pathlib import Path
import numpy as np

split_dir = Path("../outputs/splits")

test_idx_2017 = np.load(split_dir / "retained_test_idx.npy")
test_idx_2018 = np.load(split_dir / "2018_test_idx.npy")

print("2017 test rows:", len(test_idx_2017))
print("2018 test rows:", len(test_idx_2018))
print()

print("2017 test index range:", test_idx_2017.min(), "to", test_idx_2017.max())
print("2018 test index range:", test_idx_2018.min(), "to", test_idx_2018.max())

print()
print("2017 split file exists:", (split_dir / "retained_test_idx.npy").exists())
print("2018 split file exists:", (split_dir / "2018_test_idx.npy").exists())

2017 test rows: 566149
2018 test rows: 1317886

2017 test index range: 3 to 2830739
2018 test index range: 6 to 6589426

2017 split file exists: True
2018 split file exists: True


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

data_dir_2017 = Path("../data/processed/cicids17")
data_dir_2018 = Path("../data/processed/cicids18")
split_dir = Path("../outputs/splits")

test_idx_2017 = np.load(split_dir / "retained_test_idx.npy")
test_idx_2018 = np.load(split_dir / "2018_test_idx.npy")

files_2017 = sorted(data_dir_2017.glob("*.parquet"))
files_2018 = sorted(data_dir_2018.glob("*.parquet"))

rows_2017 = [len(pd.read_parquet(f, columns=["Target"])) for f in files_2017]
rows_2018 = [len(pd.read_parquet(f, columns=["Target"])) for f in files_2018]

total_2017 = sum(rows_2017)
total_2018 = sum(rows_2018)

train_rows_2017 = total_2017 - len(test_idx_2017)
train_rows_2018 = total_2018 - len(test_idx_2018)

print("2017 total rows:", total_2017)
print("2017 test rows:", len(test_idx_2017))
print("2017 training rows:", train_rows_2017)
print()

print("2018 total rows:", total_2018)
print("2018 test rows:", len(test_idx_2018))
print("2018 training rows:", train_rows_2018)
print()

assert total_2017 == 2830743
assert total_2018 == 6659532
assert train_rows_2017 == 2264594
assert train_rows_2018 == 5341646

print("Split size verification: PASSED")

2017 total rows: 2830743
2017 test rows: 566149
2017 training rows: 2264594

2018 total rows: 6659532
2018 test rows: 1317886
2018 training rows: 5341646

Split size verification: PASSED


In [4]:
from sklearn.model_selection import train_test_split

feature_cols = np.load(
    split_dir / "feature_columns_2017.npy",
    allow_pickle=True
).tolist()

df_2017_all = pd.concat(
    [pd.read_parquet(f) for f in files_2017],
    ignore_index=True
)

df_2018_all = pd.concat(
    [pd.read_parquet(f) for f in files_2018],
    ignore_index=True
)

train_mask_2017 = np.ones(len(df_2017_all), dtype=bool)
train_mask_2017[test_idx_2017] = False

train_mask_2018 = np.ones(len(df_2018_all), dtype=bool)
train_mask_2018[test_idx_2018] = False

df_2017_control_train = df_2017_all.loc[train_mask_2017].copy()
df_2018_full_train = df_2018_all.loc[train_mask_2018].copy()

df_2018_control_train, _ = train_test_split(
    df_2018_full_train,
    train_size=len(df_2017_control_train),
    stratify=df_2018_full_train["Target"],
    random_state=42
)

print("2017 controlled training rows:", len(df_2017_control_train))
print("2018 controlled training rows:", len(df_2018_control_train))
print()

print("2017 target distribution:")
print(df_2017_control_train["Target"].value_counts().sort_index())
print()

print("2018 target distribution:")
print(df_2018_control_train["Target"].value_counts().sort_index())
print()

print("2017 attack rate:",
      df_2017_control_train["Target"].mean())

print("2018 attack rate:",
      df_2018_control_train["Target"].mean())

print()
print("2017 missing values:",
      df_2017_control_train[feature_cols].isna().sum().sum())

print("2018 missing values:",
      df_2018_control_train[feature_cols].isna().sum().sum())

print()
print("Controlled training-set construction: PASSED")

2017 controlled training rows: 2264594
2018 controlled training rows: 2264594

2017 target distribution:
Target
0    1818478
1     446116
Name: count, dtype: int64

2018 target distribution:
Target
0    1813416
1     451178
Name: count, dtype: int64

2017 attack rate: 0.19699601782924445
2018 attack rate: 0.19923129708901463

2017 missing values: 4832
2018 missing values: 0

Controlled training-set construction: PASSED


In [5]:
stratify_key_2018 = (
    df_2018_full_train["Source_File"].astype(str)
    + "__"
    + df_2018_full_train["Target"].astype(str)
)

df_2018_control_train, _ = train_test_split(
    df_2018_full_train,
    train_size=len(df_2017_control_train),
    stratify=stratify_key_2018,
    random_state=42
)

print("2017 controlled training rows:", len(df_2017_control_train))
print("2018 controlled training rows:", len(df_2018_control_train))
print()

print("2017 attack rate:", df_2017_control_train["Target"].mean())
print("2018 attack rate:", df_2018_control_train["Target"].mean())
print()

print("2017 source-file counts:")
print(df_2017_control_train["Source_File"].value_counts().sort_index())
print()

print("2018 controlled source-file counts:")
print(df_2018_control_train["Source_File"].value_counts().sort_index())
print()

print("2018 controlled class distribution:")
print(df_2018_control_train["Target"].value_counts().sort_index())
print()

print("2017 controlled missing values:",
      df_2017_control_train[feature_cols].isna().sum().sum())

print("2018 controlled missing values:",
      df_2018_control_train[feature_cols].isna().sum().sum())

assert len(df_2017_control_train) == len(df_2018_control_train)
assert df_2017_control_train["Target"].isin([0, 1]).all()
assert df_2018_control_train["Target"].isin([0, 1]).all()

print()
print("CONTROLLED TRAINING SET CONSTRUCTION: PASSED")

2017 controlled training rows: 2264594
2018 controlled training rows: 2264594

2017 attack rate: 0.19699601782924445
2018 attack rate: 0.1992317386692714

2017 source-file counts:
Source_File
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv               180290
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv           231330
Friday-WorkingHours-Morning.pcap_ISCX.csv                      152232
Monday-WorkingHours.pcap_ISCX.csv                              421656
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv    233722
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv         135750
Tuesday-WorkingHours.pcap_ISCX.csv                             357665
Wednesday-workingHours.pcap_ISCX.csv                           551949
Name: count, dtype: int64

2018 controlled source-file counts:
Source_File
Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet           261720
Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet    210147
DDoS1-Tuesday-20-02-2

In [6]:
full_2018_distribution = (
    df_2018_full_train
    .groupby(["Source_File", "Target"])
    .size()
    .rename("Full_Count")
    .reset_index()
)

controlled_2018_distribution = (
    df_2018_control_train
    .groupby(["Source_File", "Target"])
    .size()
    .rename("Controlled_Count")
    .reset_index()
)

distribution_check = full_2018_distribution.merge(
    controlled_2018_distribution,
    on=["Source_File", "Target"],
    how="outer"
).fillna(0)

distribution_check["Full_%"] = (
    distribution_check["Full_Count"] /
    len(df_2018_full_train) * 100
)

distribution_check["Controlled_%"] = (
    distribution_check["Controlled_Count"] /
    len(df_2018_control_train) * 100
)

distribution_check["Absolute_Difference_%"] = (
    distribution_check["Controlled_%"] -
    distribution_check["Full_%"]
).abs()

print(distribution_check.to_string(index=False))
print()

print(
    "Maximum absolute distribution difference (%):",
    distribution_check["Absolute_Difference_%"].max()
)

print(
    "Mean absolute distribution difference (%):",
    distribution_check["Absolute_Difference_%"].mean()
)

                                                      Source_File  Target  Full_Count  Controlled_Count    Full_%  Controlled_%  Absolute_Difference_%
       Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet       0      501800            212738  9.394108      9.394090           1.804177e-05
       Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet       1      115537             48982  2.162948      2.162948           8.877988e-07
Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet       0      420423            178238  7.870664      7.870638           2.565596e-05
Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet       1       75265             31909  1.409023      1.409038           1.582920e-05
       DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet       0      303439            128643  5.680627      5.680621           6.385135e-06
       DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet       1      460523         

In [8]:
from xgboost import XGBClassifier

X_control_2017 = df_2017_control_train[feature_cols]
y_control_2017 = df_2017_control_train["Target"]

controlled_xgb_params = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "min_child_weight": 1,
    "scale_pos_weight": 1.0,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1
}

xgb_control_2017 = XGBClassifier(**controlled_xgb_params)

print("Training controlled 2017 model...")
xgb_control_2017.fit(
    X_control_2017,
    y_control_2017
)

print("Controlled 2017 model training: COMPLETE")

Training controlled 2017 model...
Controlled 2017 model training: COMPLETE


In [9]:
X_control_2018 = df_2018_control_train[feature_cols]
y_control_2018 = df_2018_control_train["Target"]

xgb_control_2018 = XGBClassifier(**controlled_xgb_params)

print("Training controlled 2018 model...")

xgb_control_2018.fit(
    X_control_2018,
    y_control_2018
)

print("Controlled 2018 model training: COMPLETE")

Training controlled 2018 model...
Controlled 2018 model training: COMPLETE


In [10]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

X_2017_test_control = df_2017_all.loc[test_idx_2017, feature_cols]
y_2017_test_control = df_2017_all.loc[test_idx_2017, "Target"]

X_2018_test_control = df_2018_all.loc[test_idx_2018, feature_cols]
y_2018_test_control = df_2018_all.loc[test_idx_2018, "Target"]

def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob)
    }

y_pred_17_to_18 = xgb_control_2017.predict(X_2018_test_control)
y_prob_17_to_18 = xgb_control_2017.predict_proba(X_2018_test_control)[:, 1]

y_pred_18_to_17 = xgb_control_2018.predict(X_2017_test_control)
y_prob_18_to_17 = xgb_control_2018.predict_proba(X_2017_test_control)[:, 1]

results_17_to_18 = evaluate_predictions(
    y_2018_test_control,
    y_pred_17_to_18,
    y_prob_17_to_18
)

results_18_to_17 = evaluate_predictions(
    y_2017_test_control,
    y_pred_18_to_17,
    y_prob_18_to_17
)

print("CONTROLLED 2017 → 2018")
print(results_17_to_18)
print()
print("Confusion Matrix:")
print(confusion_matrix(y_2018_test_control, y_pred_17_to_18))

print()
print("=" * 60)
print()

print("CONTROLLED 2018 → 2017")
print(results_18_to_17)
print()
print("Confusion Matrix:")
print(confusion_matrix(y_2017_test_control, y_pred_18_to_17))

CONTROLLED 2017 → 2018
{'Accuracy': 0.7987466290711033, 'Precision': 0.8044343376918703, 'Recall': 0.005313556139692077, 'F1': 0.010557377293804722, 'ROC-AUC': 0.7633043128088173, 'PR-AUC': 0.5559368515161836}

Confusion Matrix:
[[1051242     344]
 [ 264885    1415]]


CONTROLLED 2018 → 2017
{'Accuracy': 0.7874252184495601, 'Precision': 0.39288750698374914, 'Recall': 0.1450192773244867, 'F1': 0.211844371533167, 'ROC-AUC': 0.6713931467113403, 'PR-AUC': 0.3643221986627598}

Confusion Matrix:
[[429626  24993]
 [ 95356  16174]]


In [8]:
import gc

for name in [
    "df_2017_all",
    "df_2018_all",
    "files_2017",
    "files_2018"
]:
    if name in globals():
        del globals()[name]

gc.collect()

print("Memory cleanup completed.")

Memory cleanup completed.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_dir_2017 = Path("../data/processed/cicids17")
data_dir_2018 = Path("../data/processed/cicids18")
split_dir = Path("../outputs/splits")

feature_cols = np.load(
    split_dir / "feature_columns_2017.npy",
    allow_pickle=True
).tolist()

files_2017 = sorted(data_dir_2017.glob("*.parquet"))
files_2018 = sorted(data_dir_2018.glob("*.parquet"))

if not files_2017:
    raise FileNotFoundError(f"No 2017 parquet files found in {data_dir_2017}")

if not files_2018:
    raise FileNotFoundError(f"No 2018 parquet files found in {data_dir_2018}")

df_2017_all = pd.concat(
    [
        pd.read_parquet(f).assign(Source_File=f.name)
        for f in files_2017
    ],
    ignore_index=True
)

df_2018_all = pd.concat(
    [
        pd.read_parquet(f).assign(Source_File=f.name)
        for f in files_2018
    ],
    ignore_index=True
)

print("2017 shape:", df_2017_all.shape)
print("2018 shape:", df_2018_all.shape)
print()

print("Number of common features:", len(feature_cols))
print("First 5 features:", feature_cols[:5])
print()

print("2017 Target values:", sorted(df_2017_all["Target"].dropna().unique()))
print("2018 Target values:", sorted(df_2018_all["Target"].dropna().unique()))

2017 shape: (2830743, 71)
2018 shape: (6659532, 71)

Number of common features: 68
First 5 features: ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total']

2017 Target values: [np.int8(0), np.int8(1)]
2018 Target values: [np.int8(0), np.int8(1)]


In [2]:
import gc

test_idx_2017 = np.load(
    "../outputs/splits/retained_test_idx.npy"
)

test_idx_2018 = np.load(
    "../outputs/splits/2018_test_idx.npy"
)

def get_conflict_hashes(df, feature_columns):
    row_hash = pd.util.hash_pandas_object(
        df[feature_columns].astype(np.float64),
        index=False
    )

    temp = pd.DataFrame({
        "hash": row_hash,
        "Target": df["Target"].to_numpy()
    })

    conflict_hashes = (
        temp.groupby("hash")["Target"]
        .nunique()
    )

    conflict_hashes = conflict_hashes[
        conflict_hashes > 1
    ].index

    del temp, conflict_hashes
    gc.collect()

    return row_hash, conflict_hashes

row_hash_2017 = pd.util.hash_pandas_object(
    df_2017_all[feature_cols].astype(np.float64),
    index=False
)

conflict_summary_2017 = (
    pd.DataFrame({
        "hash": row_hash_2017,
        "Target": df_2017_all["Target"].to_numpy()
    })
    .groupby("hash")["Target"]
    .nunique()
)

conflict_hashes_2017 = conflict_summary_2017[
    conflict_summary_2017 > 1
].index

del conflict_summary_2017
gc.collect()

clean_mask_2017 = ~row_hash_2017.isin(conflict_hashes_2017)

test_mask_2017 = np.zeros(len(df_2017_all), dtype=bool)
test_mask_2017[test_idx_2017] = True

clean_test_mask_2017 = clean_mask_2017 & test_mask_2017
clean_train_mask_2017 = clean_mask_2017 & ~test_mask_2017

print("2017 conflict groups:", len(conflict_hashes_2017))
print("2017 clean rows:", clean_mask_2017.sum())
print("2017 clean train rows:", clean_train_mask_2017.sum())
print("2017 clean test rows:", clean_test_mask_2017.sum())
print()

del row_hash_2017, conflict_hashes_2017
gc.collect()

row_hash_2018 = pd.util.hash_pandas_object(
    df_2018_all[feature_cols].astype(np.float64),
    index=False
)

conflict_summary_2018 = (
    pd.DataFrame({
        "hash": row_hash_2018,
        "Target": df_2018_all["Target"].to_numpy()
    })
    .groupby("hash")["Target"]
    .nunique()
)

conflict_hashes_2018 = conflict_summary_2018[
    conflict_summary_2018 > 1
].index

del conflict_summary_2018
gc.collect()

clean_mask_2018 = ~row_hash_2018.isin(conflict_hashes_2018)

clean_positions_2018 = np.flatnonzero(clean_mask_2018)

test_orig_idx_2018 = clean_positions_2018[test_idx_2018]

test_mask_2018 = np.zeros(len(df_2018_all), dtype=bool)
test_mask_2018[test_orig_idx_2018] = True

clean_train_mask_2018 = clean_mask_2018 & ~test_mask_2018
clean_test_mask_2018 = clean_mask_2018 & test_mask_2018

print("2018 conflict groups:", len(conflict_hashes_2018))
print("2018 clean rows:", clean_mask_2018.sum())
print("2018 clean train rows:", clean_train_mask_2018.sum())
print("2018 clean test rows:", clean_test_mask_2018.sum())

del row_hash_2018, conflict_hashes_2018, clean_positions_2018
gc.collect()

2017 conflict groups: 272
2017 clean rows: 2747873
2017 clean train rows: 2193844
2017 clean test rows: 554029

2018 conflict groups: 21947
2018 clean rows: 6589430
2018 clean train rows: 5271544
2018 clean test rows: 1317886


0

In [3]:
print("2017 cleaned dataset")
print("Clean train:", clean_train_mask_2017.sum())
print("Clean test :", clean_test_mask_2017.sum())
print("Clean total:", clean_mask_2017.sum())
print()

print("2018 cleaned dataset")
print("Clean train:", clean_train_mask_2018.sum())
print("Clean test :", clean_test_mask_2018.sum())
print("Clean total:", clean_mask_2018.sum())
print()

print("2017 clean train attack rate:",
      df_2017_all.loc[clean_train_mask_2017, "Target"].mean())

print("2017 clean test attack rate:",
      df_2017_all.loc[clean_test_mask_2017, "Target"].mean())

print("2018 clean train attack rate:",
      df_2018_all.loc[clean_train_mask_2018, "Target"].mean())

print("2018 clean test attack rate:",
      df_2018_all.loc[clean_test_mask_2018, "Target"].mean())

2017 cleaned dataset
Clean train: 2193844
Clean test : 554029
Clean total: 2747873

2018 cleaned dataset
Clean train: 5271544
Clean test : 1317886
Clean total: 6589430

2017 clean train attack rate: 0.17151675324225424
2017 clean test attack rate: 0.1799165747641369
2018 clean train attack rate: 0.19795737264072916
2018 clean test attack rate: 0.19795794173395878


In [4]:
from sklearn.model_selection import train_test_split

controlled_train_size = clean_train_mask_2017.sum()

clean_train_idx_2018 = np.flatnonzero(clean_train_mask_2018)

stratify_key_2018 = (
    df_2018_all.iloc[clean_train_idx_2018]["Source_File"].astype(str)
    + "__"
    + df_2018_all.iloc[clean_train_idx_2018]["Target"].astype(str)
)

controlled_train_idx_2018, _ = train_test_split(
    clean_train_idx_2018,
    train_size=controlled_train_size,
    stratify=stratify_key_2018,
    random_state=42
)

print("2017 clean training size:", controlled_train_size)
print("2018 controlled training size:", len(controlled_train_idx_2018))

2017 clean training size: 2193844
2018 controlled training size: 2193844


In [5]:
original_2018_train = df_2018_all.iloc[clean_train_idx_2018]

controlled_2018_train = df_2018_all.iloc[controlled_train_idx_2018]

original_distribution = (
    original_2018_train
    .groupby(["Source_File", "Target"])
    .size()
    .rename("Original_Count")
)

controlled_distribution = (
    controlled_2018_train
    .groupby(["Source_File", "Target"])
    .size()
    .rename("Controlled_Count")
)

distribution_check = pd.concat(
    [original_distribution, controlled_distribution],
    axis=1
).fillna(0)

distribution_check["Original_%"] = (
    distribution_check["Original_Count"]
    / len(original_2018_train)
    * 100
)

distribution_check["Controlled_%"] = (
    distribution_check["Controlled_Count"]
    / len(controlled_2018_train)
    * 100
)

distribution_check["Difference_pp"] = (
    distribution_check["Controlled_%"]
    - distribution_check["Original_%"]
)

print(distribution_check.round(5))
print()
print(
    "Maximum absolute difference (percentage points):",
    distribution_check["Difference_pp"].abs().max()
)

                                                           Original_Count  \
Source_File                                        Target                   
Botnet-Friday-02-03-2018_TrafficForML_CICFlowMe... 0               489967   
                                                   1               115374   
Bruteforce-Wednesday-14-02-2018_TrafficForML_CI... 0               419815   
                                                   1                75358   
DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMe... 0               302837   
                                                   1               460430   
DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlow... 0               288823   
                                                   1               160359   
DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMe... 0               594028   
                                                   1                40964   
DoS2-Friday-16-02-2018_TrafficForML_CICFlowMete... 0               357231   

In [6]:
X_train_2017_control = df_2017_all.loc[
    clean_train_mask_2017, feature_cols
].to_numpy(dtype=np.float32)

y_train_2017_control = df_2017_all.loc[
    clean_train_mask_2017, "Target"
].to_numpy(dtype=np.int8)

X_test_2017_clean = df_2017_all.loc[
    clean_test_mask_2017, feature_cols
].to_numpy(dtype=np.float32)

y_test_2017_clean = df_2017_all.loc[
    clean_test_mask_2017, "Target"
].to_numpy(dtype=np.int8)

X_train_2018_control = df_2018_all.loc[
    controlled_train_idx_2018, feature_cols
].to_numpy(dtype=np.float32)

y_train_2018_control = df_2018_all.loc[
    controlled_train_idx_2018, "Target"
].to_numpy(dtype=np.int8)

X_test_2018_clean = df_2018_all.loc[
    clean_test_mask_2018, feature_cols
].to_numpy(dtype=np.float32)

y_test_2018_clean = df_2018_all.loc[
    clean_test_mask_2018, "Target"
].to_numpy(dtype=np.int8)

print("2017 controlled train:", X_train_2017_control.shape)
print("2017 clean test:", X_test_2017_clean.shape)

print("2018 controlled train:", X_train_2018_control.shape)
print("2018 clean test:", X_test_2018_clean.shape)

print()

print("2017 train missing:", np.isnan(X_train_2017_control).sum())
print("2017 test missing:", np.isnan(X_test_2017_clean).sum())
print("2018 train missing:", np.isnan(X_train_2018_control).sum())
print("2018 test missing:", np.isnan(X_test_2018_clean).sum())

print()

print("2017 train attack rate:", y_train_2017_control.mean())
print("2017 test attack rate:", y_test_2017_clean.mean())
print("2018 train attack rate:", y_train_2018_control.mean())
print("2018 test attack rate:", y_test_2018_clean.mean())

2017 controlled train: (2193844, 68)
2017 clean test: (554029, 68)
2018 controlled train: (2193844, 68)
2018 clean test: (1317886, 68)

2017 train missing: 4008
2017 test missing: 902
2018 train missing: 0
2018 test missing: 0

2017 train attack rate: 0.17151675324225424
2017 test attack rate: 0.1799165747641369
2018 train attack rate: 0.19795755760209022
2018 test attack rate: 0.19795794173395878


In [11]:
print("2017 train infinity:", np.isinf(X_train_2017_control).sum())
print("2017 test infinity:", np.isinf(X_test_2017_clean).sum())
print("2018 train infinity:", np.isinf(X_train_2018_control).sum())
print("2018 test infinity:", np.isinf(X_test_2018_clean).sum())
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

controlled_xgb_params = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "min_child_weight": 1,
    "scale_pos_weight": 1.0,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1
}

model_2017_control = XGBClassifier(
    **controlled_xgb_params
)

model_2017_control.fit(
    X_train_2017_control,
    y_train_2017_control
)

y_pred_2017_to_2018 = model_2017_control.predict(
    X_test_2018_clean
)

y_prob_2017_to_2018 = model_2017_control.predict_proba(
    X_test_2018_clean
)[:, 1]

controlled_results_2017_to_2018 = {
    "Accuracy": accuracy_score(
        y_test_2018_clean,
        y_pred_2017_to_2018
    ),
    "Precision": precision_score(
        y_test_2018_clean,
        y_pred_2017_to_2018,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test_2018_clean,
        y_pred_2017_to_2018,
        zero_division=0
    ),
    "F1": f1_score(
        y_test_2018_clean,
        y_pred_2017_to_2018,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test_2018_clean,
        y_prob_2017_to_2018
    ),
    "PR-AUC": average_precision_score(
        y_test_2018_clean,
        y_prob_2017_to_2018
    )
}

print()
print("2017 → 2018 controlled results")
for metric, value in controlled_results_2017_to_2018.items():
    print(f"{metric}: {value:.6f}")

print()
print("Confusion matrix:")
print(confusion_matrix(
    y_test_2018_clean,
    y_pred_2017_to_2018
))

2017 train infinity: 0
2017 test infinity: 0
2018 train infinity: 0
2018 test infinity: 0

2017 → 2018 controlled results
Accuracy: 0.811847
Precision: 0.964086
Recall: 0.051448
F1: 0.097683
ROC-AUC: 0.877667
PR-AUC: 0.744883

Confusion matrix:
[[1056500     500]
 [ 247464   13422]]


In [12]:
model_2018_control = XGBClassifier(
    **controlled_xgb_params
)

model_2018_control.fit(
    X_train_2018_control,
    y_train_2018_control
)

y_pred_2018_to_2017 = model_2018_control.predict(
    X_test_2017_clean
)

y_prob_2018_to_2017 = model_2018_control.predict_proba(
    X_test_2017_clean
)[:, 1]

controlled_results_2018_to_2017 = {
    "Accuracy": accuracy_score(
        y_test_2017_clean,
        y_pred_2018_to_2017
    ),
    "Precision": precision_score(
        y_test_2017_clean,
        y_pred_2018_to_2017,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test_2017_clean,
        y_pred_2018_to_2017,
        zero_division=0
    ),
    "F1": f1_score(
        y_test_2017_clean,
        y_pred_2018_to_2017,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test_2017_clean,
        y_prob_2018_to_2017
    ),
    "PR-AUC": average_precision_score(
        y_test_2017_clean,
        y_prob_2018_to_2017
    )
}

print("2018 → 2017 controlled results")
for metric, value in controlled_results_2018_to_2017.items():
    print(f"{metric}: {value:.6f}")

print()
print("Confusion matrix:")
print(confusion_matrix(
    y_test_2017_clean,
    y_pred_2018_to_2017
))

2018 → 2017 controlled results
Accuracy: 0.802866
Precision: 0.365105
Recall: 0.129506
F1: 0.191193
ROC-AUC: 0.693344
PR-AUC: 0.316603

Confusion matrix:
[[431902  22448]
 [ 86770  12909]]


In [13]:
y_pred_2017_in = model_2017_control.predict(
    X_test_2017_clean
)

y_prob_2017_in = model_2017_control.predict_proba(
    X_test_2017_clean
)[:, 1]

results_2017_in = {
    "Accuracy": accuracy_score(y_test_2017_clean, y_pred_2017_in),
    "Precision": precision_score(y_test_2017_clean, y_pred_2017_in, zero_division=0),
    "Recall": recall_score(y_test_2017_clean, y_pred_2017_in, zero_division=0),
    "F1": f1_score(y_test_2017_clean, y_pred_2017_in, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test_2017_clean, y_prob_2017_in),
    "PR-AUC": average_precision_score(y_test_2017_clean, y_prob_2017_in)
}

y_pred_2018_in = model_2018_control.predict(
    X_test_2018_clean
)

y_prob_2018_in = model_2018_control.predict_proba(
    X_test_2018_clean
)[:, 1]

results_2018_in = {
    "Accuracy": accuracy_score(y_test_2018_clean, y_pred_2018_in),
    "Precision": precision_score(y_test_2018_clean, y_pred_2018_in, zero_division=0),
    "Recall": recall_score(y_test_2018_clean, y_pred_2018_in, zero_division=0),
    "F1": f1_score(y_test_2018_clean, y_pred_2018_in, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test_2018_clean, y_prob_2018_in),
    "PR-AUC": average_precision_score(y_test_2018_clean, y_prob_2018_in)
}

print("2017 → 2017 controlled")
for metric, value in results_2017_in.items():
    print(f"{metric}: {value:.6f}")

print()
print("2018 → 2018 controlled")
for metric, value in results_2018_in.items():
    print(f"{metric}: {value:.6f}")

2017 → 2017 controlled
Accuracy: 0.999440
Precision: 0.998695
Recall: 0.998194
F1: 0.998445
ROC-AUC: 0.999996
PR-AUC: 0.999980

2018 → 2018 controlled
Accuracy: 0.985781
Precision: 0.997297
Recall: 0.930694
F1: 0.962845
ROC-AUC: 0.992663
PR-AUC: 0.983390


In [14]:
X_test_2017_retained = df_2017_all.iloc[
    test_idx_2017
][feature_cols].to_numpy(dtype=np.float32)

y_test_2017_retained = df_2017_all.iloc[
    test_idx_2017
]["Target"].to_numpy(dtype=np.int8)

y_pred_retained_2017 = model_2018_control.predict(
    X_test_2017_retained
)

y_prob_retained_2017 = model_2018_control.predict_proba(
    X_test_2017_retained
)[:, 1]

retained_results_2018_to_2017 = {
    "Accuracy": accuracy_score(
        y_test_2017_retained,
        y_pred_retained_2017
    ),
    "Precision": precision_score(
        y_test_2017_retained,
        y_pred_retained_2017,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test_2017_retained,
        y_pred_retained_2017,
        zero_division=0
    ),
    "F1": f1_score(
        y_test_2017_retained,
        y_pred_retained_2017,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test_2017_retained,
        y_prob_retained_2017
    ),
    "PR-AUC": average_precision_score(
        y_test_2017_retained,
        y_prob_retained_2017
    )
}

print("2018 → 2017 RETAINED test")
for metric, value in retained_results_2018_to_2017.items():
    print(f"{metric}: {value:.6f}")

print()
print("2018 → 2017 CLEAN test")
for metric, value in controlled_results_2018_to_2017.items():
    print(f"{metric}: {value:.6f}")

print()
print("Retained test size:", len(y_test_2017_retained))
print("Clean test size:", len(y_test_2017_clean))
print("Conflicting rows removed:", 
      len(y_test_2017_retained) - len(y_test_2017_clean))

2018 → 2017 RETAINED test
Accuracy: 0.786153
Precision: 0.365105
Recall: 0.115745
F1: 0.175768
ROC-AUC: 0.702886
PR-AUC: 0.335505

2018 → 2017 CLEAN test
Accuracy: 0.802866
Precision: 0.365105
Recall: 0.129506
F1: 0.191193
ROC-AUC: 0.693344
PR-AUC: 0.316603

Retained test size: 566149
Clean test size: 554029
Conflicting rows removed: 12120
